# ZelusBench: Sustained Attention — Medium

Isolates chain depth as the independent variable.
Controls for confounds: fixed signal-to-noise ratio (num_points = depth * 2),
POSITION-only queries, consistent transform_prob=0.1.

- Depths: [8, 10]
- 10 scenarios per depth = 20 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import time

from zelusbench.scenarios.config import ScenarioConfig, QueryType
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response

DEPTHS = [8, 9, 10]
SEEDS = 10
TOTAL = len(DEPTHS) * SEEDS
print(f"ZelusBench Sustained Attention — Medium")
print(f"Depths: {DEPTHS} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_attn_sustained_medium")
def zelusbench_attn_sustained_medium(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)
    all_scores = []
    depth_scores = {}
    scenario_num = 0
    for depth in DEPTHS:
        depth_scores[depth] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "min_chain_depth": depth, "max_chain_depth": depth,
                "num_points": depth * 2,
                "transform_prob": 0.1,
                "query_types": [QueryType.POSITION],
                "query_min_depth": max(1, depth - 2),
                "num_queries": 3,
                "seed": depth * 1000 + i,
            })
            scenario = ScenarioGenerator(cfg).generate(f"sustained_{depth}_{i}")
            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)
            avg = float(np.mean([s.score for s in scores]))
            depth_scores[depth].append(avg)
            bg = f"dim={cfg.dim} lb={cfg.leaf_bias} pts={cfg.num_points}"
            print(f"  [{scenario_num}/{TOTAL}] depth={depth} avg={avg:.2f} | {bg}")
        print(f"  >> Depth {depth} mean: {float(np.mean(depth_scores[depth])):.3f}")
    print("\n" + "=" * 60)
    for depth in DEPTHS:
        avg = float(np.mean(depth_scores[depth]))
        print(f"  depth={depth:2d}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(avg >= 0, expectation=f"Depth {depth}: valid (got {avg:.3f})")
    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {time.time()-_start:.1f}s")
    return overall, std

zelusbench_attn_sustained_medium.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_sustained_medium